In [3]:
import requests
import time
import os

# ── Configuration ──────────────────────────────────────────────────────────────
API_KEY    = "211d63664dfd4079bb1bea18cea0adef"
AUDIO_FILE = "healthvoice.mp3"
BASE_URL   = "https://api.assemblyai.com"
HEADERS    = {"authorization": API_KEY}

# ── Step 1: Upload local MP3 ───────────────────────────────────────────────────
print(f"⏫ Uploading '{AUDIO_FILE}'...")

if not os.path.exists(AUDIO_FILE):
    raise FileNotFoundError(f"❌ '{AUDIO_FILE}' not found. Place it in the same folder as this script.")

with open(AUDIO_FILE, "rb") as f:
    upload_response = requests.post(
        BASE_URL + "/v2/upload",
        headers=HEADERS,
        data=f
    )

upload_response.raise_for_status()
audio_url = upload_response.json()["upload_url"]
print("✅ Upload complete.")

# ── Step 2: Request transcription ─────────────────────────────────────────────
# FIX 1: Use "speech_models" (plural list) — not "speech_model" (deprecated singular)
# FIX 2: Nepali is ONLY supported on universal-2, not universal-3-pro
# FIX 3: Use language_detection + expected_languages to pin to Nepali
data = {
    "audio_url":          audio_url,
    "speech_models":      ["universal-2"],  # universal-2 supports Nepali
    "language_detection": True,
    "language_detection_options": {
        "expected_languages": ["ne"],       # Pin detection to Nepali only
        "fallback_language":  "ne"          # Fallback to Nepali if confidence is low
    }
}

print("📤 Requesting transcription (Nepali)...")
transcript_response = requests.post(
    BASE_URL + "/v2/transcript",
    json=data,
    headers={**HEADERS, "content-type": "application/json"}
)

# Print exact error message if request fails
if not transcript_response.ok:
    print(f"❌ Error {transcript_response.status_code}: {transcript_response.text}")
    transcript_response.raise_for_status()

transcript_id = transcript_response.json()["id"]
print(f"🆔 Transcript ID: {transcript_id}")

# ── Step 3: Poll until done ────────────────────────────────────────────────────
polling_url = BASE_URL + "/v2/transcript/" + transcript_id
print("⏳ Waiting for transcription", end="", flush=True)

while True:
    result = requests.get(polling_url, headers=HEADERS).json()

    if result["status"] == "completed":
        print("\n✅ Done!\n")
        text          = result["text"]
        detected_lang = result.get("language_code", "unknown")
        print(f"🌐 Detected language : {detected_lang}")
        break
    elif result["status"] == "error":
        raise RuntimeError(f"❌ Transcription failed: {result['error']}")
    else:
        print(".", end="", flush=True)
        time.sleep(3)

# ── Step 4: Print & Save ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📝 TRANSCRIPTION (Nepali):")
print("=" * 60)
print(text)
print("=" * 60)

with open("transcription.txt", "w", encoding="utf-8") as f:
    f.write(text)
print("\n💾 Saved to transcription.txt")

⏫ Uploading 'healthvoice.mp3'...
✅ Upload complete.
📤 Requesting transcription (Nepali)...
🆔 Transcript ID: 5e34b2b4-86cf-421b-ac8f-d5cec4913499
⏳ Waiting for transcription.
✅ Done!

🌐 Detected language : ne

📝 TRANSCRIPTION (Nepali):
मलाई दस दिन देखी जोरो आई रहे कोछा फिर पेट दुखी रहे कोछा

💾 Saved to transcription.txt
